# Telecom Cell Tower Simulator 🗼
This script generates realistic, highly correlated telecom events (Voice/Data) and sends them to NiFi via HTTP POST, acting as a real cell tower.

In [1]:
import requests
import json
import time
import random
import uuid
from datetime import datetime

# NiFi ListenHTTP Endpoint (Internal Docker Network)
NIFI_URL = "http://nifi:9090/contentListener"

def generate_event():
    # 1. Determine Network Condition
    network_condition = random.choices(['Normal', 'Congested'], weights=[80, 20], k=1)[0]
    
    if network_condition == 'Normal':
        signal_strength = random.randint(50, 75) * -1 # -50 to -75 (Good)
        latency = random.randint(20, 80)
        drop_prob = 0.02 # 2% chance of dropping
    else:
        signal_strength = random.randint(70, 95) * -1 # -70 to -95 (Weak/Congested)
        latency = random.randint(100, 300)
        drop_prob = 0.15 # 15% chance of dropping

    # 2. Determine Event Type (Voice vs Data)
    event_type = random.choices(['voice', 'data'], weights=[40, 60], k=1)[0]
    
    call_status = "completed"
    duration = 0
    data_mb = 0
    
    if event_type == 'voice':
        call_status = "dropped" if random.random() < drop_prob else "completed"
        duration = random.randint(10, 300) if call_status == 'completed' else random.randint(1, 60)
    else:
        # Data sessions rarely "drop" like calls, but let's simulate failed downloads
        call_status = "failed" if random.random() < drop_prob else "completed"
        data_mb = random.randint(1, 500)

    # 3. Build the Event JSON
    event = {
        "call_id": f"EVT_{uuid.uuid4().hex[:8]}",
        "customer_number": f"010{random.randint(10000000, 99999999)}",
        "tower_id": f"TOWER_{random.randint(1, 50)}",
        "event_type": event_type,
        "duration_seconds": duration,
        "data_consumed_mb": data_mb,
        "call_status": call_status,
        "latency_ms": latency,
        "signal_strength_dbm": signal_strength,
        "network_condition": network_condition,
        "timestamp": datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    return event

print(f"Starting Cell Tower Simulator... Sending to {NIFI_URL}")
events_sent = 0
while True:
    event_data = generate_event()
    try:
        # Send to NiFi
        response = requests.post(
            NIFI_URL, 
            json=event_data, 
            headers={'Content-Type': 'application/json'},
            timeout=2
        )
        if response.status_code == 200:
            events_sent += 1
            if events_sent % 10 == 0:
                print(f"[{datetime.now().strftime('%H:%M:%S')}] Successfully sent {events_sent} events. Last event: {event_data['network_condition']} network, {event_data['latency_ms']}ms latency.")
    except requests.exceptions.RequestException as e:
        print(f"Failed to connect to NiFi: {e}. Retrying in 5 seconds...")
        time.sleep(5)
        
    # Wait 1 second between events to simulate real-time flow
    time.sleep(1)

Starting Cell Tower Simulator... Sending to http://nifi:9090/contentListener
Failed to connect to NiFi: HTTPConnectionPool(host='nifi', port=9090): Read timed out. (read timeout=2). Retrying in 5 seconds...
[12:32:21] Successfully sent 10 events. Last event: Normal network, 29ms latency.
[12:32:31] Successfully sent 20 events. Last event: Congested network, 211ms latency.
[12:32:42] Successfully sent 30 events. Last event: Normal network, 70ms latency.
[12:32:52] Successfully sent 40 events. Last event: Normal network, 62ms latency.
[12:33:03] Successfully sent 50 events. Last event: Normal network, 39ms latency.
[12:33:14] Successfully sent 60 events. Last event: Normal network, 39ms latency.
[12:33:25] Successfully sent 70 events. Last event: Normal network, 56ms latency.
[12:33:36] Successfully sent 80 events. Last event: Normal network, 47ms latency.
[12:33:47] Successfully sent 90 events. Last event: Normal network, 76ms latency.
[12:33:57] Successfully sent 100 events. Last event

KeyboardInterrupt: 